#1. 기본 프로그램 설치

In [ ]:
!pip install google-play-scraper
!pip install pandas openpyxl
!pip install tqdm

#2. import 환경설정

In [ ]:
from google_play_scraper import reviews_all, Sort
import pandas as pd
from datetime import datetime
import os
from shutil import move
from IPython.display import display

#3. 추출 어플 환경 설정

In [ ]:
APP_ID = "co.benx.weverse"
COUNTRY = "us"
TARGET_COUNT = 5000000  # 최종 목표 리뷰 수
EXCEL_FILE = "playstore_reviews.xlsx"

# ✅ 날짜 필터 ON/OFF 설정
USE_DATE_FILTER = False  # True → 날짜 필터 적용, False → 적용 안 함
DATE_FILTER = "2025-4-08"  # 적용할 날짜

if USE_DATE_FILTER:
    filter_date = datetime.strptime(DATE_FILTER, "%Y-%m-%d")
    print(f"[INFO] {DATE_FILTER} 이후 작성된 리뷰만 수집합니다.")
else:
    print(f"[INFO] 날짜 필터 사용하지 않습니다. 전체 리뷰 사용합니다.")

#4. 리뷰 수집

In [ ]:
from tqdm import tqdm
from google_play_scraper import reviews, Sort
import pandas as pd

print(f"[INFO] 전체 리뷰 수집 시작...")

all_reviews = []
page = 1
batch_size = 100
token = None

with tqdm(total=TARGET_COUNT, desc="리뷰 수집 진행중", unit="개") as pbar:
    while len(all_reviews) < TARGET_COUNT:
        result, continuation = reviews(
            APP_ID,
            lang=COUNTRY,
            country=COUNTRY,
            sort=Sort.NEWEST,
            count=batch_size,
            continuation_token=token
        )

        if not result:
            break

        for review in result:
            review_date = review['at']
            if USE_DATE_FILTER:
                if review_date >= filter_date:
                    all_reviews.append(review)
            else:
                all_reviews.append(review)

        token = continuation
        pbar.update(len(result))
        page += 1

print(f"[INFO] 리뷰 수집 완료! 총 {len(all_reviews)}개")

# ==========================
# 📦 DataFrame 변환
# ==========================
df = pd.DataFrame(all_reviews)

# ==========================
# 🔁 중복 제거 (userName + at 기준)
# ==========================
before = len(df)
df.drop_duplicates(subset=["userName", "at"], inplace=True)
after = len(df)
print(f"[INFO] 중복 제거 완료: {before} → {after}개")

# ==========================
# 🧹 필요 없는 컬럼 제거
# ==========================
columns_to_drop = ["userImage", "reviewCreatedVersion", "replyContent", "repliedAt", "appVersion"]
existing_columns = [col for col in columns_to_drop if col in df.columns]
df.drop(columns=existing_columns, inplace=True)
print(f"[INFO] 필요 없는 컬럼 제거 완료: {existing_columns}")

# ==========================
# 🎯 목표 개수 맞추기
# ==========================
if after >= TARGET_COUNT:
    df = df.iloc[:TARGET_COUNT]
    print("리뷰 크롤링 완료!")
else:
    print(f"[⚠️ WARNING] 목표 개수에 도달하지 못했습니다. 최종 {after}개 리뷰만 확보.")

rename_map = {
    "content": "comment",
    "score": "score",
    "at": "date",
    "userName": "username",
    "thumbsUpCount": "likes count"
}
df.rename(columns=rename_map, inplace=True)

df.to_excel(EXCEL_FILE, index=False)
print(f"[✅ 저장 완료] '{EXCEL_FILE}'에 최종 {len(df)}개 리뷰 저장됨!")

from google.colab import files
files.download(EXCEL_FILE)


#5. 전처리: 이모지, 짧은 리뷰 데이터 제거

In [ ]:
!pip uninstall -y pandas numpy
!pip install pandas numpy --upgrade --force-reinstall
!pip install nltk emoji

In [ ]:
import pandas as pd
import re
import emoji
from tqdm import tqdm
from google.colab import files

# 파일 업로드
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# tqdm과 pandas 연동
tqdm.pandas()

# 엑셀 파일 읽기
df = pd.read_excel(file_name)

# 이모지 + 비영어 문자 제거 함수
def clean_text(text):
    if isinstance(text, float):
        text = str(text)
    text = emoji.replace_emoji(text, replace="")
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.strip()

# 전처리 적용
df['cleaned_comment'] = df['comment'].progress_apply(clean_text)

# x단어 이하 리뷰 제거
df['word_count'] = df['cleaned_comment'].apply(lambda x: len(x.split()))
df = df[df['word_count'] > 0].reset_index(drop=True)

# 🔄 열 이름 변경
df = df.rename(columns={
    'comment': 'previous comment',        # 원래 comment → previous comment
    'cleaned_comment': 'comment'          # 전처리된 comment → comment
})

# 👉 word_count 열은 불필요하면 제거
df = df.drop(columns=['word_count'])

# ✅ 엑셀로 저장
preprocessed_file_path = 'preprocessed_comments.xlsx'
df.to_excel(preprocessed_file_path, index=False)
print(f"✅ 전처리된 데이터를 '{preprocessed_file_path}' 파일로 저장했습니다.")

# 다운로드
files.download(preprocessed_file_path)


#6. 랜덤 Sampling

In [ ]:
import pandas as pd
from google.colab import files

# 엑셀 파일 업로드
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# 엑셀 파일 불러오기
df = pd.read_excel(file_name)

# 전체 개수 확인
print(f"전체 데이터 수: {len(df)}개")

# 샘플링 개수 입력
n = int(input("몇 개를 랜덤 샘플링할까요? 숫자를 입력하세요: "))

# 유효성 검사
if n > len(df):
    raise ValueError("❌ 샘플 수가 전체 데이터보다 많습니다.")

# 랜덤 샘플링
sampled_df = df.sample(n=n, random_state=42).reset_index(drop=True)

# 샘플링 결과 저장
sampled_file = f"random_sample_{n}.xlsx"
sampled_df.to_excel(sampled_file, index=False)

print(f"✅ {n}개의 랜덤 샘플 데이터를 '{sampled_file}' 파일로 저장했습니다.")
files.download(sampled_file)


#7. 토픽모델링(BERTopic)

In [ ]:
pip install pandas openpyxl bertopic tqdm hdbscan plotly umap-learn matplotlib python-louvain

In [ ]:
# 1. 라이브러리 임포트
import pandas as pd
from bertopic import BERTopic
from tqdm import tqdm
import re
import string
from google.colab import files
import hdbscan
import plotly.io as pio
pio.renderers.default = "colab"  # plotly 시각화 설정

# 2. 파일 업로드
print("📁 엑셀 파일을 업로드 해주세요")
uploaded = files.upload()

# 3. 파일 읽기
for fn in uploaded.keys():
    file_path = fn

df = pd.read_excel(file_path)
print(f"✅ 파일 로드 완료: {file_path}")
print("📊 comment 열 미리보기:")
print(df["comment"].head())

# 4. 텍스트 전처리 함수
def clean_text(text):
    text = text.lower()
    text = re.sub(rf"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# 5. 텍스트 전처리
comments = df["comment"].dropna().astype(str).tolist()
print("🧹 텍스트 전처리 중...")
comments_cleaned = [clean_text(text) for text in tqdm(comments)]

# 6. hDBSCAN 설정
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=30,
    min_samples=5,
    metric='euclidean',
    prediction_data=True
)

# 7. BERTopic 모델 학습
print("🤖 BERTopic + hDBSCAN 훈련 중...")
topic_model = BERTopic(hdbscan_model=hdbscan_model, language="english", verbose=True)
topics, probs = topic_model.fit_transform(comments_cleaned)

# 8. 시각화
fig = topic_model.visualize_topics()
fig.show()

# 9. 원본 데이터에 토픽 ID 붙이기
df_result = df.copy()
df_result["topic_id"] = topics

# 🔽 토픽 ID 기준 오름차순 정렬
df_result = df_result.sort_values(by="topic_id").reset_index(drop=True)

# 10. 저장 파일 이름 정의 및 저장
output_file = "bertopic_result.xlsx"
df_result.to_excel(output_file, index=False)

# 11. 다운로드 제공
print("💾 분석 결과를 엑셀로 저장했습니다. 다운로드 버튼을 누르세요.")
files.download(output_file)


In [ ]:
!pip install umap-learn matplotlib

In [ ]:
# 🧠 1. topic_id별 대표 키워드 출력
print("🧠 각 topic_id별 대표 키워드:")
for topic_id in topic_model.get_topics().keys():
    if topic_id == -1:
        continue
    print(f"\n🟦 Topic {topic_id}:")
    for word, weight in topic_model.get_topic(topic_id)[:10]:
        print(f"  - {word} ({weight:.4f})")

# 🧠 2. 토픽 임베딩을 이용한 유사도 행렬 수동 생성
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 토픽 임베딩 추출 (각 토픽을 임베딩한 벡터)
embeddings = topic_model.topic_embeddings_

# 유사도 행렬 생성 (cosine similarity)
similarity_matrix = cosine_similarity(embeddings)

# 🧠 3. 시각화 (히트맵)
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 10))
sns.heatmap(similarity_matrix, cmap="YlGnBu")
plt.title("Topic Similarity Heatmap (Cosine Similarity)")
plt.xlabel("Topic ID")
plt.ylabel("Topic ID")
plt.show()


In [ ]:
# 1. 직접 SentenceTransformer 선언 (BERTopic 학습 시 사용한 것과 동일한 모델)
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")  # BERTopic 기본 모델

# 2. 문장 임베딩 벡터 생성
embeddings = embedding_model.encode(comments_cleaned, show_progress_bar=True)

# 3. UMAP 차원 축소
from umap import UMAP
import matplotlib.pyplot as plt
import numpy as np

umap_model = UMAP(n_components=2, random_state=42)
embeddings_2d = umap_model.fit_transform(embeddings)

# 4. 시각화
unique_topics = sorted(set(topics))
colors = plt.cm.get_cmap('tab20', len(unique_topics))

plt.figure(figsize=(10, 8))
for topic in unique_topics:
    idx = [i for i, t in enumerate(topics) if t == topic]
    plt.scatter(embeddings_2d[idx, 0], embeddings_2d[idx, 1],
                label=f"Topic {topic}", alpha=0.5, s=30, color=colors(topic))

plt.title("🔍 문장 클러스터 시각화 (UMAP + Topic Color)")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.grid(True)
plt.show()


#8. 토픽모델링 (Louvain, META Topic 추출(임계값조정가능, 높을수록세분화))

In [ ]:
!pip install python-louvain

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
import community.community_louvain as community_louvain
from tqdm import tqdm
import numpy as np

# 0. 모델 준비
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# 1. 토픽별 평균 임베딩 생성
print("📌 [1/5] 토픽별 평균 임베딩 생성 중...")
topic_ids = sorted(set(t for t in topics if t != -1))
topic_vectors = []

for tid in tqdm(topic_ids, desc="▶️ 토픽 임베딩 중"):
    idx = [i for i, t in enumerate(topics) if t == tid]
    topic_sentences = [comments_cleaned[i] for i in idx]
    if topic_sentences:
        topic_embedding = embedding_model.encode(topic_sentences)
        topic_mean = np.mean(topic_embedding, axis=0)
        topic_vectors.append(topic_mean)
    else:
        topic_vectors.append(np.zeros(embedding_model.get_sentence_embedding_dimension()))

# 2. 유사도 행렬 계산
print("🔗 [2/5] 토픽 간 유사도 계산 중...")
similarity_matrix = cosine_similarity(topic_vectors)

# 3. 토픽 간 그래프 생성
print("🔗 [3/5] 유사도 기반 그래프 생성 중...")
G = nx.Graph()
for i in tqdm(range(len(topic_ids)), desc="▶️ 그래프 엣지 생성 중"):
    for j in range(i + 1, len(topic_ids)):
        weight = similarity_matrix[i][j]

        if weight > 0.5:  # 임계값은 필요 시 조절 가능, 기본은 0.3
          G.add_edge(topic_ids[i], topic_ids[j], weight=weight)

# 4. Louvain community detection
print("🧠 [4/5] Louvain 커뮤니티 탐지 중...")
partition = community_louvain.best_partition(G)

# 💡 고립된 토픽도 메타토픽으로 넣기
isolated_topics = list(set(topic_ids) - set(partition.keys()))
if isolated_topics:
    next_meta_topic = max(partition.values()) + 1 if partition else 0
    for iso_tid in isolated_topics:
        partition[iso_tid] = next_meta_topic
        next_meta_topic += 1

# 5. 메타토픽 정리 및 출력
print("📁 [5/5] 메타토픽 정리 중...")
meta_topic_dict = {}
for topic_id, community_id in partition.items():
    if community_id not in meta_topic_dict:
        meta_topic_dict[community_id] = []
    meta_topic_dict[community_id].append(topic_id)

# 결과 출력
print("\n🧠 Louvain 기반 메타토픽 그룹:")
for group, topics_in_group in meta_topic_dict.items():
    print(f"\n🔷 메타토픽 {group}:")
    print(f"Topic IDs: {topics_in_group}")
    for tid in topics_in_group:
        words = topic_model.get_topic(tid)[:5]
        keywords = ", ".join([w[0] for w in words])
        print(f"  - Topic {tid}: {keywords}")


In [ ]:
import pandas as pd

# 원본 데이터에 topic_id 추가
df_with_topic = df.copy()
df_with_topic["topic_id"] = topics

# topic_id → meta_topic_id 매핑
topic_to_meta = partition  # Louvain 결과

# topic_id 기준으로 meta_topic_id 컬럼 생성
df_with_topic["meta_topic"] = df_with_topic["topic_id"].map(topic_to_meta)

# -1 (노이즈) 제거
df_with_topic = df_with_topic[df_with_topic["topic_id"] != -1]

# 원하는 순서로 정렬
df_with_topic = df_with_topic.sort_values(by=["meta_topic", "topic_id"]).reset_index(drop=True)

# 결과 미리보기
print("📄 메타토픽별 리뷰 데이터:")
print(df_with_topic[["meta_topic", "topic_id", "comment"]].head())

# 엑셀로 저장
output_file = "meta_topic_grouped_reviews.xlsx"
df_with_topic.to_excel(output_file, index=False)

# 다운로드
from google.colab import files
files.download(output_file)


In [ ]:
# 📌 토픽-메타토픽-키워드(+빈도수) 테이블 생성
rows = []

for topic_id, meta_topic in partition.items():
    if topic_id == -1:
        continue  # 노이즈 제외

    # 각 topic_id에서 상위 10개 키워드 및 가중치(빈도수 유사값) 추출
    keywords = topic_model.get_topic(topic_id)

    for word, weight in keywords[:10]:
        rows.append({
            "meta_topic": meta_topic,
            "topic_id": topic_id,
            "keyword": word,
            "score": round(weight, 4)  # score 또는 weight로 표시
        })

# 데이터프레임으로 변환
df_keywords = pd.DataFrame(rows)

# 정렬
df_keywords = df_keywords.sort_values(by=["meta_topic", "topic_id", "score"], ascending=[True, True, False]).reset_index(drop=True)

# 엑셀로 저장 (선택적)
output_keywords_file = "meta_topic_keywords_with_scores.xlsx"
df_keywords.to_excel(output_keywords_file, index=False)

# 다운로드 (Colab 환경에서만 사용)
from google.colab import files
files.download(output_keywords_file)


#9. 감성분석(RoBERTa)

In [ ]:
!pip install transformers torch pandas openpyxl tqdm scipy

In [ ]:
# 📚 라이브러리
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from tqdm import tqdm
from google.colab import files

# ✅ 디바이스 설정 (GPU 있으면 사용)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 📂 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# 📄 데이터 로딩
df = pd.read_excel(filename)
if 'comment' not in df.columns:
    raise ValueError("❌ 'comment' 열이 없습니다.")
df['comment'] = df['comment'].fillna("")

# 🤖 모델 로드
MODEL = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL).to(device)

# ✅ 감성 분석 함수 (batch 처리)
def analyze_sentiment_batch(texts):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    probs = softmax(outputs.logits.cpu().numpy(), axis=1)
    return probs  # shape: (batch_size, 3)

# ✅ 배치 감성 분석 실행
batch_size = 32
scores = []

for i in tqdm(range(0, len(df), batch_size)):
    batch_texts = df['comment'].iloc[i:i+batch_size].tolist()
    try:
        probs = analyze_sentiment_batch(batch_texts)
        for prob in probs:
            pos = prob[2]
            neg = prob[0]
            neu = prob[1]
            # compound 계산식: (pos - neg) * (1 - neu)
            compound = round((pos - neg) * (1 - neu), 4)
            scores.append(compound)
    except Exception as e:
        print(f"❌ 오류 발생 at batch {i}: {e}")
        scores.extend([0.0] * len(batch_texts))

# ✅ 결과 저장
df['sentiment_score'] = scores
output_file = filename.replace(".xlsx", "_roberta_sentiment_score_batch.xlsx")
df.to_excel(output_file, index=False)
files.download(output_file)

print("✅ 배치 기반 RoBERTa 감성 분석 완료! 정확하고 빠르게 처리됐습니다.")

#9. 임의 주제의 문장 관련성 수치 측정(Zero-shot classification)

In [ ]:
# 📦 라이브러리 설치
!pip install scipy transformers pandas openpyxl tqdm matplotlib datasets scikit-learn --quiet

Meta topic 0:  Language Accessibility

In [ ]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
from google.colab import files

# ⬆️ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

# ✅ 'comment' 열 확인
if 'comment' not in df.columns:
    raise ValueError("❌ 'comment' 열이 엑셀에 없습니다!")

# 🤖 Zero-shot 분류기 로드
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

#  관련성 라벨 정의
chatbot_labels = [
    ("Completely Unrelated", 0.0),
    ("Slightly Unrelated", 0.25),
    ("Neutral", 0.5),
    ("Somewhat Related", 0.75),
    ("Strongly Related", 1.0)
]

# n 관련성 점수 계산 함수
def classify_chatbot_score(comment):
    if pd.isna(comment) or not str(comment).strip():
        return 0.0

    labels = [label for label, _ in chatbot_labels]

    output = classifier(
        comment,
        candidate_labels=labels,
        hypothesis_template="This comment is {} to the Language Accessibility.", # 해당 프롬프트 수정해서 사용
        return_all_scores=True
    )

    # output 검증 및 가중 평균 계산
    if isinstance(output, list):
        result = output[0]  # return_all_scores=True 이므로 list of list
    elif isinstance(output, dict) and "scores" in output:
        result = [
            {'label': label, 'score': score}
            for label, score in zip(output["labels"], output["scores"])
        ]
    else:
        raise ValueError("❌ 모델 출력 형식이 예상과 다릅니다")

    weighted = sum(r['score'] * w for r, (_, w) in zip(result, chatbot_labels))
    return round(weighted, 4)

# 🏃 분석 실행
tqdm.pandas()
df["Chatbot_score"] = df["comment"].progress_apply(classify_chatbot_score)

# 💾 저장 및 다운로드
output_file = filename.replace(".xlsx", "_chatbot_scored.xlsx")
df.to_excel(output_file, index=False)
files.download(output_file)

print("✅ 완료: Chatbot_score (0~1) 계산 완료")


Meta topic 1:  User Experience

In [ ]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
from google.colab import files

# ⬆️ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

# ✅ 'comment' 열 확인
if 'comment' not in df.columns:
    raise ValueError("❌ 'comment' 열이 엑셀에 없습니다!")

# 🤖 Zero-shot 분류기 로드
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

#  관련성 라벨 정의
chatbot_labels = [
    ("Completely Unrelated", 0.0),
    ("Slightly Unrelated", 0.25),
    ("Neutral", 0.5),
    ("Somewhat Related", 0.75),
    ("Strongly Related", 1.0)
]

# n 관련성 점수 계산 함수
def classify_chatbot_score(comment):
    if pd.isna(comment) or not str(comment).strip():
        return 0.0

    labels = [label for label, _ in chatbot_labels]

    output = classifier(
        comment,
        candidate_labels=labels,
        hypothesis_template="This comment is {} to the User Experience.", # 해당 프롬프트 수정해서 사용
        return_all_scores=True
    )

    # output 검증 및 가중 평균 계산
    if isinstance(output, list):
        result = output[0]  # return_all_scores=True 이므로 list of list
    elif isinstance(output, dict) and "scores" in output:
        result = [
            {'label': label, 'score': score}
            for label, score in zip(output["labels"], output["scores"])
        ]
    else:
        raise ValueError("❌ 모델 출력 형식이 예상과 다릅니다")

    weighted = sum(r['score'] * w for r, (_, w) in zip(result, chatbot_labels))
    return round(weighted, 4)

# 🏃 분석 실행
tqdm.pandas()
df["Chatbot_score"] = df["comment"].progress_apply(classify_chatbot_score)

# 💾 저장 및 다운로드
output_file = filename.replace(".xlsx", "_chatbot_scored.xlsx")
df.to_excel(output_file, index=False)
files.download(output_file)

print("✅ 완료: Chatbot_score (0~1) 계산 완료")


Meta topic 2:  BTS Fans' Reactions

In [ ]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
from google.colab import files

# ⬆️ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

# ✅ 'comment' 열 확인
if 'comment' not in df.columns:
    raise ValueError("❌ 'comment' 열이 엑셀에 없습니다!")

# 🤖 Zero-shot 분류기 로드
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

#  관련성 라벨 정의
chatbot_labels = [
    ("Completely Unrelated", 0.0),
    ("Slightly Unrelated", 0.25),
    ("Neutral", 0.5),
    ("Somewhat Related", 0.75),
    ("Strongly Related", 1.0)
]

# n 관련성 점수 계산 함수
def classify_chatbot_score(comment):
    if pd.isna(comment) or not str(comment).strip():
        return 0.0

    labels = [label for label, _ in chatbot_labels]

    output = classifier(
        comment,
        candidate_labels=labels,
        hypothesis_template="This comment is {} to the BTS Fans' Reactions.", # 해당 프롬프트 수정해서 사용
        return_all_scores=True
    )

    # output 검증 및 가중 평균 계산
    if isinstance(output, list):
        result = output[0]  # return_all_scores=True 이므로 list of list
    elif isinstance(output, dict) and "scores" in output:
        result = [
            {'label': label, 'score': score}
            for label, score in zip(output["labels"], output["scores"])
        ]
    else:
        raise ValueError("❌ 모델 출력 형식이 예상과 다릅니다")

    weighted = sum(r['score'] * w for r, (_, w) in zip(result, chatbot_labels))
    return round(weighted, 4)

# 🏃 분석 실행
tqdm.pandas()
df["Chatbot_score"] = df["comment"].progress_apply(classify_chatbot_score)

# 💾 저장 및 다운로드
output_file = filename.replace(".xlsx", "_chatbot_scored.xlsx")
df.to_excel(output_file, index=False)
files.download(output_file)

print("✅ 완료: Chatbot_score (0~1) 계산 완료")


Meta topic 3:  Connecting with Idols

In [ ]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
from google.colab import files

# ⬆️ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

# ✅ 'comment' 열 확인
if 'comment' not in df.columns:
    raise ValueError("❌ 'comment' 열이 엑셀에 없습니다!")

# 🤖 Zero-shot 분류기 로드
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

#  관련성 라벨 정의
chatbot_labels = [
    ("Completely Unrelated", 0.0),
    ("Slightly Unrelated", 0.25),
    ("Neutral", 0.5),
    ("Somewhat Related", 0.75),
    ("Strongly Related", 1.0)
]

# n 관련성 점수 계산 함수
def classify_chatbot_score(comment):
    if pd.isna(comment) or not str(comment).strip():
        return 0.0

    labels = [label for label, _ in chatbot_labels]

    output = classifier(
        comment,
        candidate_labels=labels,
        hypothesis_template="This comment is {} to the Connecting with Idols.", # 해당 프롬프트 수정해서 사용
        return_all_scores=True
    )

    # output 검증 및 가중 평균 계산
    if isinstance(output, list):
        result = output[0]  # return_all_scores=True 이므로 list of list
    elif isinstance(output, dict) and "scores" in output:
        result = [
            {'label': label, 'score': score}
            for label, score in zip(output["labels"], output["scores"])
        ]
    else:
        raise ValueError("❌ 모델 출력 형식이 예상과 다릅니다")

    weighted = sum(r['score'] * w for r, (_, w) in zip(result, chatbot_labels))
    return round(weighted, 4)

# 🏃 분석 실행
tqdm.pandas()
df["Chatbot_score"] = df["comment"].progress_apply(classify_chatbot_score)

# 💾 저장 및 다운로드
output_file = filename.replace(".xlsx", "_chatbot_scored.xlsx")
df.to_excel(output_file, index=False)
files.download(output_file)

print("✅ 완료: Chatbot_score (0~1) 계산 완료")


Meta topic 4:  Filipino Users' Reviews

In [ ]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
from google.colab import files

# ⬆️ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

# ✅ 'comment' 열 확인
if 'comment' not in df.columns:
    raise ValueError("❌ 'comment' 열이 엑셀에 없습니다!")

# 🤖 Zero-shot 분류기 로드
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

#  관련성 라벨 정의
chatbot_labels = [
    ("Completely Unrelated", 0.0),
    ("Slightly Unrelated", 0.25),
    ("Neutral", 0.5),
    ("Somewhat Related", 0.75),
    ("Strongly Related", 1.0)
]

# n 관련성 점수 계산 함수
def classify_chatbot_score(comment):
    if pd.isna(comment) or not str(comment).strip():
        return 0.0

    labels = [label for label, _ in chatbot_labels]

    output = classifier(
        comment,
        candidate_labels=labels,
        hypothesis_template="This comment is {} to the Filipino Users' Reviews.", # 해당 프롬프트 수정해서 사용
        return_all_scores=True
    )

    # output 검증 및 가중 평균 계산
    if isinstance(output, list):
        result = output[0]  # return_all_scores=True 이므로 list of list
    elif isinstance(output, dict) and "scores" in output:
        result = [
            {'label': label, 'score': score}
            for label, score in zip(output["labels"], output["scores"])
        ]
    else:
        raise ValueError("❌ 모델 출력 형식이 예상과 다릅니다")

    weighted = sum(r['score'] * w for r, (_, w) in zip(result, chatbot_labels))
    return round(weighted, 4)

# 🏃 분석 실행
tqdm.pandas()
df["Chatbot_score"] = df["comment"].progress_apply(classify_chatbot_score)

# 💾 저장 및 다운로드
output_file = filename.replace(".xlsx", "_chatbot_scored.xlsx")
df.to_excel(output_file, index=False)
files.download(output_file)

print("✅ 완료: Chatbot_score (0~1) 계산 완료")


#10. 회귀분석

In [ ]:
pip install pandas numpy scikit-learn openpyxl


X -> Y

In [ ]:
# 📚 라이브러리 불러오기
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from google.colab import files

# ✅ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ✅ 엑셀 데이터 로딩
df = pd.read_excel(filename)

# ✅ 컬럼 확인
print("\n✅ 데이터셋 컬럼 목록:")
for idx, col in enumerate(df.columns):
    print(f"{idx}: {col}")

# ✅ 사용자 입력: 종속 변수
target_column = input("\n🎯 종속변수로 사용할 컬럼명을 입력하세요: ")

# ✅ 사용자 입력: 독립 변수
input_columns = input("\n📌 독립변수로 사용할 컬럼명을 쉼표(,)로 구분해서 입력하세요:\n👉 ")
independent_columns = [col.strip() for col in input_columns.split(",")]

# ✅ X, y 정의
X = df[independent_columns]
y = df[target_column]

# ✅ 공분산 행렬 출력
print("\n📊 [공분산 행렬] (독립변수 간 상관 정도)")
print(X.cov())

# ✅ 상관계수 행렬 출력
print("\n📈 [상관계수 행렬] (독립변수 간 상관 정도)")
print(X.corr())

# ✅ 상관계수가 0.7 이상인 변수쌍만 추출
print("\n📌 [상관계수 ≥ 0.7 인 변수쌍 목록]")
corr_matrix = X.corr()
high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        var1 = corr_matrix.columns[i]
        var2 = corr_matrix.columns[j]
        corr_value = corr_matrix.iloc[i, j]
        if abs(corr_value) >= 0.7:
            high_corr_pairs.append((var1, var2, corr_value))

if high_corr_pairs:
    for var1, var2, corr in high_corr_pairs:
        print(f"{var1} ↔ {var2}: 상관계수 = {corr:.3f}")
else:
    print("✅ 0.7 이상인 변수쌍 없음")

# ✅ VIF 계산
print("\n📌 [VIF (분산팽창계수) 결과]")
X_vif = sm.add_constant(X)  # 절편 포함
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print(vif_data)

# ✅ 회귀 분석 실행
model = sm.OLS(y, X_vif).fit()

# ✅ 회귀 분석 요약 출력
print("\n📈 [회귀분석 결과 요약]")
print(model.summary())

# ✅ 유의성 표시용 별 생성 함수
def pval_to_stars(p):
    if p <= 0.001:
        return "***"
    elif p <= 0.01:
        return "**"
    elif p <= 0.05:
        return "*"
    else:
        return ""

# ✅ 회귀 계수 요약표 생성
summary_df = pd.DataFrame({
    "Variable": model.params.index,
    "Coefficient": model.params.values,
    "P-value": model.pvalues.values,
    "Significance": model.pvalues.apply(pval_to_stars)
})

# ✅ 출력
print("\n📋 회귀 계수 + 유의성(* 표시) 요약표:")
print(summary_df.to_string(index=False))
print(f"\n📌 Adjusted R² (조정된 설명력): {model.rsquared_adj:.4f}")


표준화 베타값 추가

In [ ]:
# 📚 라이브러리 불러오기
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
from google.colab import files

# ✅ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ✅ 엑셀 데이터 로딩
df = pd.read_excel(filename)

# ✅ 컬럼 확인
print("\n✅ 데이터셋 컬럼 목록:")
for idx, col in enumerate(df.columns):
    print(f"{idx}: {col}")

# ✅ 사용자 입력: 종속 변수
target_column = input("\n🎯 종속변수로 사용할 컬럼명을 입력하세요: ")

# ✅ 사용자 입력: 독립 변수
input_columns = input("\n📌 독립변수로 사용할 컬럼명을 쉼표(,)로 구분해서 입력하세요:\n👉 ")
independent_columns = [col.strip() for col in input_columns.split(",")]

# ✅ X, y 정의
X = df[independent_columns]
y = df[target_column]

# ✅ 공분산 행렬 출력
print("\n📊 [공분산 행렬] (독립변수 간 상관 정도)")
print(X.cov())

# ✅ 상관계수 행렬 출력
print("\n📈 [상관계수 행렬] (독립변수 간 상관 정도)")
print(X.corr())

# ✅ 상관계수가 0.7 이상인 변수쌍만 추출
print("\n📌 [상관계수 ≥ 0.7 인 변수쌍 목록]")
corr_matrix = X.corr()
high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        var1 = corr_matrix.columns[i]
        var2 = corr_matrix.columns[j]
        corr_value = corr_matrix.iloc[i, j]
        if abs(corr_value) >= 0.7:
            high_corr_pairs.append((var1, var2, corr_value))

if high_corr_pairs:
    for var1, var2, corr in high_corr_pairs:
        print(f"{var1} ↔ {var2}: 상관계수 = {corr:.3f}")
else:
    print("✅ 0.7 이상인 변수쌍 없음")

# ✅ VIF 계산
print("\n📌 [VIF (분산팽창계수) 결과]")
X_vif = sm.add_constant(X)  # 절편 포함
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print(vif_data)

# ✅ 회귀 분석 실행 (비표준화)
model = sm.OLS(y, X_vif).fit()

# ✅ 표준화된 베타 계산용 모델 (표준화 후)
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten()

X_scaled_const = sm.add_constant(X_scaled)
model_std = sm.OLS(y_scaled, X_scaled_const).fit()

# ✅ 유의성 표시용 별 생성 함수
def pval_to_stars(p):
    if p <= 0.001:
        return "***"
    elif p <= 0.01:
        return "**"
    elif p <= 0.05:
        return "*"
    else:
        return ""

# ✅ 회귀 계수 요약표 생성 (소수점 셋째 자리까지 반올림)
summary_df = pd.DataFrame({
    "Variable": model.params.index,
    "Coefficient": np.round(model.params.values, 3),
    "P-value": np.round(model.pvalues.values, 3),
    "Significance": model.pvalues.apply(pval_to_stars),
    "Standardized Beta": np.round(model_std.params, 3)  # <- 여기만 수정
})


# ✅ 출력
print("\n📋 회귀 계수 + 유의성(* 표시) + 표준화 베타 요약표:")
print(summary_df.to_string(index=False))
print(f"\n📌 Adjusted R² (조정된 설명력): {model.rsquared_adj:.4f}")


In [ ]:
# ✅ 결과들을 엑셀로 저장
from google.colab import files

# 📌 상관계수 0.7 이상 변수쌍 DataFrame 생성
if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs, columns=["변수1", "변수2", "상관계수"])
else:
    high_corr_df = pd.DataFrame([["없음", "없음", "N/A"]], columns=["변수1", "변수2", "상관계수"])

# 📌 회귀분석 요약 텍스트 저장용
model_summary_text = model.summary().as_text()
model_summary_df = pd.DataFrame(model_summary_text.splitlines(), columns=["회귀분석 요약"])

# ✅ ExcelWriter 사용하여 시트별 저장
output_filename = "회귀분석_전체결과.xlsx"
with pd.ExcelWriter(output_filename) as writer:
    X.cov().to_excel(writer, sheet_name="공분산 행렬")
    X.corr().to_excel(writer, sheet_name="상관계수 행렬")
    high_corr_df.to_excel(writer, sheet_name="상관 높은 변수쌍", index=False)
    vif_data.to_excel(writer, sheet_name="VIF 결과", index=False)
    summary_df.to_excel(writer, sheet_name="회귀 계수 요약", index=False)
    model_summary_df.to_excel(writer, sheet_name="회귀분석 텍스트", index=False)

# ✅ 파일 다운로드
files.download(output_filename)


X -> M -> Y

In [ ]:
# 📚 라이브러리 불러오기
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from google.colab import files
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ✅ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ✅ 엑셀 데이터 로딩
df = pd.read_excel(filename)

# ✅ 컬럼 목록 출력
print("\n✅ 데이터셋 컬럼 목록:")
for idx, col in enumerate(df.columns):
    print(f"{idx}: {col}")

# ✅ 사용자 입력
target_column = input("\n🎯 종속변수로 사용할 컬럼명을 입력하세요: ")
input_columns = input("\n📌 독립변수로 사용할 컬럼명을 쉼표(,)로 구분해서 입력하세요:\n👉 ")
independent_columns = [col.strip() for col in input_columns.split(",")]
mediator_column = input("\n🧩 매개변수(중재변수)로 사용할 컬럼명을 입력하세요:\n👉 ")

# ✅ 변수 설정
X = df[independent_columns]
M = df[[mediator_column]]
y = df[target_column]

# ✅ 공분산 행렬 출력
print("\n📊 [공분산 행렬]")
print(pd.concat([X, M, y], axis=1).cov())

# ✅ 표준화 함수
def standardize(df):
    return pd.DataFrame(StandardScaler().fit_transform(df), columns=df.columns)

# ✅ 유의성 표시 함수
def significance_stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    elif p < 0.1:
        return '.'
    else:
        return ''

def print_coef_with_significance(model):
    summary_df = pd.DataFrame({
        'Coef': model.params,
        'p-value': model.pvalues,
        'Signif': model.pvalues.apply(significance_stars)
    })
    print(summary_df)

# ✅ 표준화된 데이터
X_std = standardize(X)
M_std = standardize(M)
y_std = standardize(y.to_frame())

# ✅ 1단계: X ➝ y
X1_const = sm.add_constant(X)
model1 = sm.OLS(y, X1_const).fit()

X1_std_const = sm.add_constant(X_std)
model1_std = sm.OLS(y_std, X1_std_const).fit()

print("\n🔹 [1단계: 독립변수 ➝ 종속변수]")
print(model1.summary())
print(f"\n📌 Adjusted R²: {model1.rsquared_adj:.4f}")
print("\n📊 Standardized Betas with significance:")
print_coef_with_significance(model1_std)

# ✅ 2단계: X ➝ M
model2 = sm.OLS(M, X1_const).fit()
model2_std = sm.OLS(M_std, X1_std_const).fit()

print("\n🔹 [2단계: 독립변수 ➝ 매개변수]")
print(model2.summary())
print(f"\n📌 Adjusted R²: {model2.rsquared_adj:.4f}")
print("\n📊 Standardized Betas with significance:")
print_coef_with_significance(model2_std)

# ✅ 3단계: X + M ➝ y
XM = pd.concat([X, M], axis=1)
XM_const = sm.add_constant(XM)
model3 = sm.OLS(y, XM_const).fit()

XM_std = pd.concat([X_std, M_std], axis=1)
XM_std_const = sm.add_constant(XM_std)
model3_std = sm.OLS(y_std, XM_std_const).fit()

print("\n🔹 [3단계: 독립변수 + 매개변수 ➝ 종속변수]")
print(model3.summary())
print(f"\n📌 Adjusted R²: {model3.rsquared_adj:.4f}")
print("\n📊 Standardized Betas with significance:")
print_coef_with_significance(model3_std)

# ✅ VIF 계산 함수
def compute_vif(df_with_const):
    vif_df = pd.DataFrame()
    vif_df["Variable"] = df_with_const.columns
    vif_df["VIF"] = [variance_inflation_factor(df_with_const.values, i)
                     for i in range(df_with_const.shape[1])]
    return vif_df

# ✅ VIF 출력
print("\n📌 [VIF (다중공선성 지표)]")
vif_results = compute_vif(XM_const)
print(vif_results)

# ✅ 위계회귀 검정 (R² 증가량의 유의성 검정)
anova_results = sm.stats.anova_lm(model1, model3)

anova_results_rounded = anova_results.copy()
anova_results_rounded['F'] = anova_results_rounded['F'].apply(lambda x: f"{x:.4f}" if pd.notnull(x) else "")
anova_results_rounded['Pr(>F)'] = anova_results_rounded['Pr(>F)'].apply(lambda x: f"{x:.4f}" if pd.notnull(x) else "")

print("\n📈 [위계회귀: R² 증가 유의성 검정 (ANOVA 비교)]")
print(anova_results_rounded)


2단계 3단계 VIF 값 추가

In [ ]:
# 📚 라이브러리 불러오기
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from google.colab import files
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ✅ 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ✅ 엑셀 데이터 로딩
df = pd.read_excel(filename)

# ✅ 컬럼 목록 출력
print("\n✅ 데이터셋 컬럼 목록:")
for idx, col in enumerate(df.columns):
    print(f"{idx}: {col}")

# ✅ 사용자 입력
target_column = input("\n🎯 종속변수로 사용할 컬럼명을 입력하세요: ")
input_columns = input("\n📌 독립변수로 사용할 컬럼명을 쉼표(,)로 구분해서 입력하세요:\n👉 ")
independent_columns = [col.strip() for col in input_columns.split(",")]
mediator_column = input("\n🧩 매개변수(중재변수)로 사용할 컬럼명을 입력하세요:\n👉 ")

# ✅ 변수 설정
X = df[independent_columns]
M = df[[mediator_column]]
y = df[target_column]

# ✅ 공분산 행렬 출력
print("\n📊 [공분산 행렬]")
print(pd.concat([X, M, y], axis=1).cov())

# ✅ 표준화 함수
def standardize(df):
    return pd.DataFrame(StandardScaler().fit_transform(df), columns=df.columns)

# ✅ 유의성 표시 함수
def significance_stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    elif p < 0.1:
        return '.'
    else:
        return ''

def print_coef_with_significance(model):
    summary_df = pd.DataFrame({
        'Coef': model.params,
        'p-value': model.pvalues,
        'Signif': model.pvalues.apply(significance_stars)
    })
    print(summary_df)

# ✅ 표준화된 데이터
X_std = standardize(X)
M_std = standardize(M)
y_std = standardize(y.to_frame())

# ✅ 1단계: X ➝ y
X1_const = sm.add_constant(X)
model1 = sm.OLS(y, X1_const).fit()

X1_std_const = sm.add_constant(X_std)
model1_std = sm.OLS(y_std, X1_std_const).fit()

print("\n🔹 [1단계: 독립변수 ➝ 종속변수]")
print(model1.summary())
print(f"\n📌 Adjusted R²: {model1.rsquared_adj:.4f}")
print("\n📊 Standardized Betas with significance:")
print_coef_with_significance(model1_std)

# ✅ 2단계: X ➝ M
model2 = sm.OLS(M, X1_const).fit()
model2_std = sm.OLS(M_std, X1_std_const).fit()

print("\n🔹 [2단계: 독립변수 ➝ 매개변수]")
print(model2.summary())
print(f"\n📌 Adjusted R²: {model2.rsquared_adj:.4f}")
print("\n📊 Standardized Betas with significance:")
print_coef_with_significance(model2_std)

# ✅ 2단계 VIF 출력
def compute_vif(df_with_const):
    vif_df = pd.DataFrame()
    vif_df["Variable"] = df_with_const.columns
    vif_df["VIF"] = [variance_inflation_factor(df_with_const.values, i)
                     for i in range(df_with_const.shape[1])]
    return vif_df

print("\n📌 [2단계 VIF (다중공선성 지표)]")
vif_results_2nd = compute_vif(X1_const)
print(vif_results_2nd)

# ✅ 3단계: X + M ➝ y
XM = pd.concat([X, M], axis=1)
XM_const = sm.add_constant(XM)
model3 = sm.OLS(y, XM_const).fit()

XM_std = pd.concat([X_std, M_std], axis=1)
XM_std_const = sm.add_constant(XM_std)
model3_std = sm.OLS(y_std, XM_std_const).fit()

print("\n🔹 [3단계: 독립변수 + 매개변수 ➝ 종속변수]")
print(model3.summary())
print(f"\n📌 Adjusted R²: {model3.rsquared_adj:.4f}")
print("\n📊 Standardized Betas with significance:")
print_coef_with_significance(model3_std)

# ✅ 3단계 VIF 출력
print("\n📌 [VIF (다중공선성 지표)]")
vif_results = compute_vif(XM_const)
print(vif_results)

# ✅ 위계회귀 검정 (R² 증가량의 유의성 검정)
anova_results = sm.stats.anova_lm(model1, model3)

anova_results_rounded = anova_results.copy()
anova_results_rounded['F'] = anova_results_rounded['F'].apply(lambda x: f"{x:.4f}" if pd.notnull(x) else "")
anova_results_rounded['Pr(>F)'] = anova_results_rounded['Pr(>F)'].apply(lambda x: f"{x:.4f}" if pd.notnull(x) else "")

print("\n📈 [위계회귀: R² 증가 유의성 검정 (ANOVA 비교)]")
print(anova_results_rounded)


In [ ]:
!pip install xlsxwriter

In [ ]:
# ✅ 표준화된 회귀계수 출력 함수 (DataFrame 반환)
def print_coef_with_significance_df(model):
    return pd.DataFrame({
        'Coef': model.params,
        'p-value': model.pvalues,
        'Significance': model.pvalues.apply(significance_stars)
    })

# ✅ 출력용 객체 정리
output_dict = {
    "Covariance Matrix": pd.concat([X, M, y], axis=1).cov(),
    "Step1_Standardized": print_coef_with_significance_df(model1_std),
    "Step2_Standardized": print_coef_with_significance_df(model2_std),
    "Step3_Standardized": print_coef_with_significance_df(model3_std),
    "Hierarchical_Regression_ANOVA": anova_results
}

output_filename = "mediation_analysis_results.xlsx"

import xlsxwriter  # 설치 안 되어 있으면 '!pip install xlsxwriter' 실행 필요

with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:
    # ✅ 공분산 행렬 저장
    output_dict["Covariance Matrix"].to_excel(writer, sheet_name="Covariance Matrix")

    # ✅ 1단계 summary 텍스트 저장
    worksheet = writer.book.add_worksheet('Step1_Summary')
    writer.sheets['Step1_Summary'] = worksheet
    for i, line in enumerate(model1.summary().as_text().split('\n')):
        worksheet.write(i, 0, line)

    # ✅ 2단계 summary 텍스트 저장
    worksheet = writer.book.add_worksheet('Step2_Summary')
    writer.sheets['Step2_Summary'] = worksheet
    for i, line in enumerate(model2.summary().as_text().split('\n')):
        worksheet.write(i, 0, line)

    # ✅ 3단계 summary 텍스트 저장
    worksheet = writer.book.add_worksheet('Step3_Summary')
    writer.sheets['Step3_Summary'] = worksheet
    for i, line in enumerate(model3.summary().as_text().split('\n')):
        worksheet.write(i, 0, line)

    # ✅ 표준화 회귀계수 저장
    output_dict["Step1_Standardized"].to_excel(writer, sheet_name="Step1_Std_Betas")
    output_dict["Step2_Standardized"].to_excel(writer, sheet_name="Step2_Std_Betas")
    output_dict["Step3_Standardized"].to_excel(writer, sheet_name="Step3_Std_Betas")

    # ✅ 위계회귀 결과 저장
    output_dict["Hierarchical_Regression_ANOVA"].to_excel(writer, sheet_name="Hierarchical_ANOVA")

# ✅ 파일 다운로드
from google.colab import files
files.download(output_filename)
